# Seed and length sweep

Re-run the induction-head identification and ablation from `02_induction_heads.ipynb` over a grid of block lengths and random seeds, then report the mean and standard deviation across seeds at each length, plus how consistently the same head ranks first. This notebook builds on `01_observe_copying.ipynb` and `02_induction_heads.ipynb`; it does not repeat their single-run analyses.

## Metric

Each (length, seed) cell reuses the paired copying benefit (control minus repeated-context loss on the second block) under three conditions: no ablation, the top-`k` induction heads zero-ablated, and an equal number of random control heads zero-ablated. Top heads are re-identified in every cell rather than fixed in advance.

The **standard deviation across seeds** is the sample standard deviation (ddof=1) of the per-cell benefit at a fixed length. It differs from the within-run standard error reported in earlier notebooks, which only reflects variation across sequence pairs for a single seed. Across-seed variation also includes the choice of control heads, which is seeded.

**Top-1 consistency** is the fraction of seeds at a given length in which the same head has the highest induction score. A value of 1.0 means every seed agreed.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root / "src"))
from copy_lab.sweep import run


## Experimental design

Grid: block lengths 8, 16, 32, 64 crossed with seeds 0 to 4, with 16 sequence pairs per cell, top-2 induction heads ablated, and CPU execution. That is 20 cells and roughly 1,900 single-sequence forward passes; expect well under a few minutes on CPU.

Each seed changes the sampled token sequences and the randomly chosen control heads. It does not change the model weights.

Rerunning overwrites the files in `results/sweep/`; preserve each run separately when comparing configurations.

In [ ]:
summary = run()


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(root / "results/sweep/sweep.png")))
print("length  n_seeds  baseline  induction_ablated  control_ablated  top-1 head(s)")
for row in summary["per_length"]:
    def fmt(stats):
        std = "n/a" if stats["std"] is None else f"{stats['std']:.2f}"
        return f"{stats['mean']:.2f} +/- {std}"
    benefit = row["benefit"]
    print(
        f"{row['length']:>6}  {row['n_seeds']:>7}  {fmt(benefit['baseline'])}  "
        f"{fmt(benefit['induction_ablated'])}  {fmt(benefit['control_ablated'])}  "
        f"{row['top1_head_counts']}"
    )


## Interpretation and limitations

If the induction-ablated benefit stays far below both the baseline and the control-ablated benefit at every length, and the same head ranks first in every seed, the conclusion from the single-run notebook is not an artifact of one seed or one block length.

Seeds vary the sampled tokens and the control-head choice, not the trained model, so this does not measure variation across training runs. Five seeds give a coarse estimate of the standard deviation. Zero-ablation is off-distribution, so effect sizes are evidence of contribution rather than precise causal estimates. Random token inputs still differ from natural language.

## References

- [TransformerLens main demo](https://transformerlensorg.github.io/TransformerLens/generated/demos/Main_Demo.html): pretrained models and repeated-sequence induction analysis.
- [ARENA transformer interpretability](https://learn.arena.education/chapter1_transformer_interp/02_intro_mech_interp/): induction circuits and experimental methods.